# ระบบยืมหนังสือ — Source Code & Demo
**CP352301 | 1.3 Source Code · 1.4 Demo**

> รัน cell ตามลำดับจากบนลงล่าง: **A (Data) → B (Books) → C (Members) → D (Borrow) → E (Return) → F (Report) → Demo → Menu → Tests**

---
## 1.3 Source Code พร้อมคำอธิบาย


In [ ]:
# ============================================================
#  LIBRARY BORROWING SYSTEM  |  library_system.py
#  CP352301 Python Fundamentals Mini Project
# ============================================================
from datetime import date, timedelta

# ── DATA LAYER ───────────────────────────────────────────────
# Concept: dict เพราะค้นหาด้วย key ได้เร็ว O(1)
books      = {}   # { ISBN: {title, author, total, available} }
members    = {}   # { member_id: {name, email, borrowed_books:[]} }
borrow_log = []   # append-only log -- ไม่ลบ เพิ่มอย่างเดียว
_log_id    = 0

def seed_data():
    """โหลดข้อมูลตัวอย่างสำหรับ demo"""
    for isbn, title, author, qty in [
        ("ISBN001","Learning Python",           "Mark Lutz",         3),
        ("ISBN002","Clean Code",                "Robert C. Martin",  2),
        ("ISBN003","The Pragmatic Programmer",  "David Thomas",      2),
        ("ISBN004","Python Crash Course",       "Eric Matthes",      4),
        ("ISBN005","Automate the Boring Stuff", "Al Sweigart",       3),
    ]:
        add_book(isbn, title, author, qty, verbose=False)
    for mid, name, email in [
        ("M001","Alice Wonderland","alice@kkumail.com"),
        ("M002","Bob Builder",     "bob@kkumail.com"),
        ("M003","Craig Mack",      "craig@kkumail.com"),
    ]:
        register_member(mid, name, email, verbose=False)
    print("Seed data loaded: 5 books | 3 members")

print("Library System ready. Call seed_data() to start.")


In [ ]:
# ── BOOK FUNCTIONS ──────────────────────────────────────────
def add_book(isbn: str, title: str, author: str,
             qty: int = 1, verbose: bool = True) -> dict:
    """เพิ่มหนังสือ หรืออัปเดตจำนวนถ้ามีอยู่แล้ว (upsert)

    Args:
        isbn:    รหัสหนังสือ (unique key)
        title:   ชื่อหนังสือ
        author:  ผู้แต่ง
        qty:     จำนวนเล่มที่เพิ่ม (default 1)
        verbose: แสดงข้อความยืนยัน (default True)
    Returns:
        dict ข้อมูลหนังสือ
    Raises:
        ValueError: isbn/title ว่าง หรือ qty < 1
    """
    if not isbn or not title:
        raise ValueError("ISBN และชื่อหนังสือต้องไม่ว่าง")
    if qty < 1:
        raise ValueError(f"จำนวนต้องมากกว่า 0 (ได้รับ: {qty})")

    if isbn in books:                       # UPDATE
        books[isbn]["total"]     += qty
        books[isbn]["available"] += qty
        action = "อัปเดต"
    else:                                   # INSERT
        books[isbn] = {"title": title, "author": author,
                       "total": qty, "available": qty}
        action = "เพิ่ม"

    if verbose:
        print(f"[OK] {action}: [{isbn}] {title} ({qty} เล่ม)")
    return books[isbn]


def list_books(only_available: bool = False) -> None:
    """แสดงรายการหนังสือทั้งหมด หรือเฉพาะที่ว่าง"""
    if not books:
        print("ยังไม่มีหนังสือในระบบ"); return
    label = "ที่ว่างอยู่" if only_available else "ทั้งหมด"
    print(f"\n--- รายการหนังสือ{label} ---")
    print(f"{'ISBN':<10} {'ชื่อหนังสือ':<32} {'ผู้แต่ง':<22} ว่าง/ทั้งหมด")
    print("-" * 75)
    shown = 0
    for isbn, b in books.items():
        if only_available and b["available"] == 0:
            continue
        status = "[V]" if b["available"] > 0 else "[X]"
        print(f"{isbn:<10} {b['title']:<32} {b['author']:<22} "
              f"{status} {b['available']}/{b['total']}")
        shown += 1
    print(f"-" * 75)
    print(f"รวม {shown} รายการ")


def search_book(keyword: str) -> list:
    """ค้นหาด้วย title หรือ author (case-insensitive)"""
    kw = keyword.lower()
    # list comprehension กรองจาก dict
    results = [(isbn, b) for isbn, b in books.items()
               if kw in b["title"].lower() or kw in b["author"].lower()]
    if results:
        print(f"\nค้นหา '{keyword}' -- พบ {len(results)} รายการ:")
        for isbn, b in results:
            avail = "ว่าง" if b["available"] > 0 else "ไม่ว่าง"
            print(f"  [{isbn}] {b['title']} -- {b['author']} ({avail})")
    else:
        print(f"ไม่พบ '{keyword}'")
    return results


In [ ]:
# ── MEMBER FUNCTIONS ────────────────────────────────────────
def register_member(member_id: str, name: str, email: str,
                    verbose: bool = True) -> dict:
    """ลงทะเบียนสมาชิกใหม่

    Raises:
        ValueError: ถ้า member_id ซ้ำหรือ field ว่าง
    """
    if not member_id or not name:
        raise ValueError("รหัสสมาชิกและชื่อต้องไม่ว่าง")
    if member_id in members:
        raise ValueError(f"รหัสสมาชิก '{member_id}' มีอยู่แล้ว")

    members[member_id] = {
        "name":           name,
        "email":          email,
        "borrowed_books": [],          # list ของ ISBN ที่ยืมอยู่
        "joined_date":    str(date.today()),
    }
    if verbose:
        print(f"[OK] ลงทะเบียน: [{member_id}] {name}")
    return members[member_id]


def list_members() -> None:
    """แสดงรายชื่อสมาชิกทั้งหมด"""
    if not members:
        print("ยังไม่มีสมาชิก"); return
    print(f"\n--- สมาชิกทั้งหมด ({len(members)} คน) ---")
    for mid, m in members.items():
        n = len(m["borrowed_books"])
        print(f"  {mid:<8} {m['name']:<25} ยืมอยู่ {n} เล่ม")


In [ ]:
# ── BORROW FUNCTION ─────────────────────────────────────────
# Data Flow:
#   borrow_book():
#     books[isbn]["available"]        -= 1
#     members[member_id]["borrowed_books"].append(isbn)
#     borrow_log.append({ log entry })

def borrow_book(isbn: str, member_id: str) -> dict:
    """ยืมหนังสือ — มี 4 validation ก่อน mutate data

    Returns: dict log entry ที่สร้างขึ้น
    Raises:
        KeyError:   ถ้าไม่พบ isbn หรือ member_id
        ValueError: ถ้าหนังสือไม่ว่าง หรือยืมซ้ำ
    """
    global _log_id

    # validation 1: ISBN มีในระบบไหม
    if isbn not in books:
        raise KeyError(f"ไม่พบหนังสือ ISBN: {isbn}")
    book = books[isbn]

    # validation 2: หนังสือว่างไหม
    if book["available"] < 1:
        raise ValueError(
            f"'{book['title']}' ถูกยืมหมดแล้ว (มี {book['total']} เล่ม)")

    # validation 3: สมาชิกมีในระบบไหม
    if member_id not in members:
        raise KeyError(f"ไม่พบสมาชิก ID: {member_id}")
    member = members[member_id]

    # validation 4: ยืมซ้ำไหม
    if isbn in member["borrowed_books"]:
        raise ValueError(
            f"'{member['name']}' ยืม '{book['title']}' อยู่แล้ว")

    # ── mutate data (เฉพาะหลัง validation ผ่านทั้งหมด) ──────────
    book["available"] -= 1
    member["borrowed_books"].append(isbn)
    _log_id += 1
    borrow_date = date.today()
    due_date    = borrow_date + timedelta(days=14)  # คืนใน 14 วัน

    entry = {
        "log_id":        _log_id,
        "isbn":          isbn,
        "member_id":     member_id,
        "borrow_date":   str(borrow_date),
        "due_date":      str(due_date),
        "returned_date": None,   # None = ยังไม่คืน
    }
    borrow_log.append(entry)

    print(f"[OK] ยืมสำเร็จ! (Log #{_log_id})")
    print(f"     หนังสือ : {book['title']}")
    print(f"     สมาชิก  : {member['name']}")
    print(f"     ยืม     : {borrow_date}  |  กำหนดคืน: {due_date}")
    return entry


In [ ]:
# ── RETURN FUNCTION ─────────────────────────────────────────
# Data Flow:
#   return_book():
#     entry["returned_date"]              = today  (อัปเดต)
#     books[isbn]["available"]            += 1
#     members[member_id]["borrowed_books"].remove(isbn)

def return_book(isbn: str, member_id: str) -> dict:
    """คืนหนังสือ

    Returns: dict log entry ที่อัปเดตแล้ว
    Raises:  KeyError ถ้าไม่พบรายการยืมที่ยังค้างอยู่
    """
    if isbn not in books:
        raise KeyError(f"ไม่พบหนังสือ ISBN: {isbn}")
    if member_id not in members:
        raise KeyError(f"ไม่พบสมาชิก ID: {member_id}")

    # next() + generator expression -- หา log entry แรกที่ยังไม่คืน
    # Concept: next(generator, default) -- คืน default ถ้าหาไม่เจอ
    entry = next(
        (e for e in borrow_log
         if e["isbn"] == isbn
         and e["member_id"] == member_id
         and e["returned_date"] is None),
        None
    )
    if entry is None:
        raise KeyError(
            f"ไม่พบรายการยืม -- {members[member_id]['name']} "
            f"ไม่ได้ยืม ISBN:{isbn} อยู่ในขณะนี้")

    # mutate data
    today = date.today()
    entry["returned_date"] = str(today)
    books[isbn]["available"] += 1
    members[member_id]["borrowed_books"].remove(isbn)

    book = books[isbn]
    days = (today - date.fromisoformat(entry["borrow_date"])).days
    overdue_days = max(0, days - 14)
    fine = overdue_days * 5  # 5 บาท/วัน

    print(f"[OK] คืนสำเร็จ!")
    print(f"     หนังสือ : {book['title']}")
    print(f"     สมาชิก  : {members[member_id]['name']}")
    print(f"     ยืม: {entry['borrow_date']} | คืน: {today} ({days} วัน)")
    if overdue_days > 0:
        print(f"     [!] เกินกำหนด {overdue_days} วัน -- ค่าปรับ {fine} บาท")
    return entry


def my_loans(member_id: str) -> list:
    """ดูรายการหนังสือที่สมาชิกยืมอยู่ตอนนี้"""
    if member_id not in members:
        raise KeyError(f"ไม่พบสมาชิก ID: {member_id}")
    member = members[member_id]
    active = [e for e in borrow_log
              if e["member_id"] == member_id and e["returned_date"] is None]
    print(f"\n-- รายการยืมของ {member['name']} --")
    if not active:
        print("  ไม่มีหนังสือที่ยืมอยู่")
    else:
        for e in active:
            b = books.get(e["isbn"], {})
            today = date.today()
            overdue = today > date.fromisoformat(e["due_date"])
            status = "[!] เกิน" if overdue else "[V] ปกติ"
            print(f"  #{e['log_id']:<3} {e['isbn']:<10} "
                  f"{b.get('title','?'):<32} due:{e['due_date']} {status}")
    return active


In [ ]:
# ── REPORT FUNCTION ─────────────────────────────────────────
def view_report() -> None:
    """รายงานสรุปภาพรวม"""
    total_books     = sum(b["total"]     for b in books.values())
    available_books = sum(b["available"] for b in books.values())
    returned_logs   = sum(1 for e in borrow_log if e["returned_date"])
    active_logs     = len(borrow_log) - returned_logs

    # นับจำนวนครั้งที่ยืมต่อ ISBN
    borrow_count = {}
    for e in borrow_log:
        borrow_count[e["isbn"]] = borrow_count.get(e["isbn"], 0) + 1
    popular = sorted(borrow_count.items(), key=lambda x: -x[1])[:3]

    line = "=" * 50
    print(f"\n{line}")
    print(f"  LIBRARY REPORT")
    print(f"{line}")
    print(f"  หนังสือในระบบ    : {len(books):>3} ชื่อ ({total_books} เล่ม)")
    print(f"  พร้อมให้ยืม      : {available_books:>3} เล่ม")
    print(f"  กำลังถูกยืม      : {total_books-available_books:>3} เล่ม")
    print(f"  สมาชิก           : {len(members):>3} คน")
    print(f"  รายการยืมทั้งหมด : {len(borrow_log):>3}")
    print(f"  ยืมอยู่ตอนนี้    : {active_logs:>3}")
    print(f"  คืนแล้ว          : {returned_logs:>3}")
    if popular:
        print(f"\n  -- หนังสือยอดนิยม --")
        for i, (isbn, cnt) in enumerate(popular, 1):
            t = books.get(isbn, {}).get("title", "?")
            print(f"  {i}. {t} ({cnt} ครั้ง)")
    print(f"{line}")


---
## 1.4 Demo
### Happy Run

In [ ]:
# DEMO 1: ตั้งค่าระบบ
books.clear(); members.clear(); borrow_log.clear()
seed_data()
print()
list_books()


In [ ]:
# DEMO 2: ค้นหาหนังสือ
search_book("python")


In [ ]:
# DEMO 3: ยืมหนังสือ -- Happy Path
print("=== Alice ยืม Learning Python ===")
borrow_book("ISBN001", "M001")
print("\n=== Bob ยืม Clean Code ===")
borrow_book("ISBN002", "M002")
print("\n=== Craig ยืม Python Crash Course ===")
borrow_book("ISBN004", "M003")


In [ ]:
# DEMO 4: ดูรายการยืมของ Alice
my_loans("M001")


In [ ]:
# DEMO 5: Alice คืนหนังสือ
return_book("ISBN001", "M001")


In [ ]:
# DEMO 6: รายงานสรุป
view_report()


### App Walk-through — ระบบเมนู Interactive

In [ ]:
# ── MAIN MENU ───────────────────────────────────────────────
# Concept: dispatcher dict -- map choice string -> function
# เหมือน Flask route decorator แต่ simplify สำหรับ CLI

def main_menu():
    """ระบบเมนูหลัก -- รัน interactive CLI"""
    MENU = {
        "1": ("เพิ่มหนังสือ",          _ui_add_book),
        "2": ("แสดงหนังสือทั้งหมด",   lambda: list_books()),
        "3": ("ค้นหาหนังสือ",          _ui_search),
        "4": ("ลงทะเบียนสมาชิก",       _ui_register),
        "5": ("แสดงสมาชิก",            lambda: list_members()),
        "6": ("ยืมหนังสือ",            _ui_borrow),
        "7": ("คืนหนังสือ",            _ui_return),
        "8": ("รายการยืมของฉัน",       _ui_my_loans),
        "9": ("รายงานสรุป",            lambda: view_report()),
        "0": ("ออกจากระบบ",            None),
    }
    print("\n" + "="*40)
    print("  ระบบยืมหนังสือ  --  Library System")
    print("="*40)
    while True:
        print("\n  เมนูหลัก:")
        for k, (label, _) in MENU.items():
            print(f"  [{k}] {label}")
        choice = input("\n  เลือก: ").strip()
        if choice == "0":
            print("  ลาก่อน!"); break
        elif choice in MENU:
            try:
                MENU[choice][1]()
            except (KeyError, ValueError) as e:
                print(f"  [ERR] {e}")
        else:
            print("  กรุณาเลือก 0-9")

def _ui_add_book():
    isbn   = input("  ISBN: ").strip().upper()
    title  = input("  ชื่อหนังสือ: ").strip()
    author = input("  ผู้แต่ง: ").strip()
    qty    = int(input("  จำนวน: ").strip() or "1")
    add_book(isbn, title, author, qty)

def _ui_search():
    kw = input("  ค้นหา: ").strip()
    search_book(kw)

def _ui_register():
    mid   = input("  รหัสสมาชิก: ").strip().upper()
    name  = input("  ชื่อ-สกุล: ").strip()
    email = input("  อีเมล: ").strip()
    register_member(mid, name, email)

def _ui_borrow():
    isbn = input("  ISBN: ").strip().upper()
    mid  = input("  รหัสสมาชิก: ").strip().upper()
    borrow_book(isbn, mid)

def _ui_return():
    isbn = input("  ISBN: ").strip().upper()
    mid  = input("  รหัสสมาชิก: ").strip().upper()
    return_book(isbn, mid)

def _ui_my_loans():
    mid = input("  รหัสสมาชิก: ").strip().upper()
    my_loans(mid)

print("main_menu() พร้อมแล้ว")
print("รัน: main_menu()  (ใช้ใน terminal หรือ Colab Runtime)")


### Testing — ทดสอบ Edge Cases ครบ

In [ ]:
# TEST SUITE
books.clear(); members.clear(); borrow_log.clear()
seed_data()

passed = 0
failed = 0

def run_test(name, fn, expect_pass=True):
    global passed, failed
    try:
        fn()
        if expect_pass:
            print(f"  PASS  {name}"); passed += 1
        else:
            print(f"  FAIL  {name}  (ควร raise แต่ไม่ raise)"); failed += 1
    except Exception as e:
        if not expect_pass:
            print(f"  PASS  {name}  --> {type(e).__name__}: {e}"); passed += 1
        else:
            print(f"  FAIL  {name}  --> {type(e).__name__}: {e}"); failed += 1

print("=" * 55)
print("  TEST SUITE -- Library System")
print("=" * 55)

print("\n  Book Tests")
run_test("add_book ปกติ",
    lambda: add_book("ISBNTEST","Test","Author",2,False))
run_test("add_book qty=0 --> ValueError",
    lambda: add_book("X","X","X",0,False), expect_pass=False)
run_test("search_book พบผลลัพธ์",
    lambda: assert_result(search_book("python")))

print("\n  Member Tests")
run_test("register_member ปกติ",
    lambda: register_member("M099","Test","t@t.com",False))
run_test("register ซ้ำ --> ValueError",
    lambda: register_member("M001","Dup","d@d.com",False), expect_pass=False)

print("\n  Borrow Tests")
run_test("borrow ปกติ",
    lambda: borrow_book("ISBN001","M001"))
run_test("borrow ISBN ไม่มี --> KeyError",
    lambda: borrow_book("XXXXX","M001"), expect_pass=False)
run_test("borrow สมาชิกไม่มี --> KeyError",
    lambda: borrow_book("ISBN002","M999"), expect_pass=False)
run_test("borrow ซ้ำ --> ValueError",
    lambda: borrow_book("ISBN001","M001"), expect_pass=False)

print("\n  Return Tests")
run_test("return ปกติ",
    lambda: return_book("ISBN001","M001"))
run_test("return ไม่มีรายการ --> KeyError",
    lambda: return_book("ISBN001","M001"), expect_pass=False)
run_test("return ISBN ไม่มี --> KeyError",
    lambda: return_book("XXXXX","M001"), expect_pass=False)

print(f"\n{'='*55}")
print(f"  ผลการทดสอบ: PASS {passed}  |  FAIL {failed}")
print(f"{'='*55}")

def assert_result(results):
    assert len(results) > 0


In [ ]:
# TEST: Full Lifecycle (ยืมจนหมด -> ยืมไม่ได้ -> คืน -> ยืมได้อีก)
print("\n  TEST: Full Lifecycle")
print("-" * 45)
books.clear(); members.clear(); borrow_log.clear()
books["ISBNX"]  = {"title":"Popular Book","author":"A","total":1,"available":1}
members["MA"] = {"name":"Alice","email":"a@a.com","borrowed_books":[],"joined_date":"2026-08-03"}
members["MB"] = {"name":"Bob",  "email":"b@b.com","borrowed_books":[],"joined_date":"2026-08-03"}

print("\n[1] Alice ยืม (ว่าง 1) --> สำเร็จ")
borrow_book("ISBNX","MA")

print("\n[2] Bob ยืม (ว่าง 0) --> ValueError")
try:    borrow_book("ISBNX","MB")
except ValueError as e: print(f"  [OK] {e}")

print("\n[3] Alice คืน")
return_book("ISBNX","MA")

print("\n[4] Bob ยืม (ว่างแล้ว 1) --> สำเร็จ")
borrow_book("ISBNX","MB")
print("\n  Full Lifecycle PASSED!")


In [ ]:
# TEST: Overdue Fine
print("\n  TEST: Overdue Fine (เกินกำหนด)")
print("-" * 45)
books.clear(); members.clear(); borrow_log.clear()
seed_data()
borrow_book("ISBN003","M002")

# แฮ็ควันที่เพื่อ simulate เกิน 20 วัน
last = borrow_log[-1]
past = date.today() - timedelta(days=20)
last["borrow_date"] = str(past)
last["due_date"]    = str(past + timedelta(days=14))

print("\nคืนหลังเกิน 6 วัน (20-14=6):")
return_book("ISBN003","M002")
print("  ค่าปรับที่ถูกต้อง = 6 x 5 = 30 บาท  [PASS]")
